In [ ]:
# ============================================================
# IMPORTS
# ============================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
import re
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 250)

try:
    from IPython.display import display
except Exception:
    display = print


Mounted at /content/drive


In [ ]:
# ============================================================
# 0. PATHS
# ============================================================

BASE          = "/content/drive/MyDrive/Dubai_Real_Estate_Data"
SOCIAL_PARQUET = f"{BASE}/final_pipeline_outputs/master_processed.parquet"
SOCIAL_CSV     = f"{BASE}/final_pipeline_outputs/master_processed.csv"
DLD_RAW_PATH   = f"{BASE}/dld/transactions-2026-06-19.csv"
OUT_DIR        = f"{BASE}/final_pipeline_outputs/dld_social_correlation"

os.makedirs(OUT_DIR, exist_ok=True)
print("Output directory:", OUT_DIR)

if not os.path.exists(SOCIAL_PARQUET) and not os.path.exists(SOCIAL_CSV):
    raise FileNotFoundError(
        "Could not find master_processed.parquet or master_processed.csv. "
        "Run v5 export first."
    )
if not os.path.exists(DLD_RAW_PATH):
    raise FileNotFoundError(f"Could not find DLD file: {DLD_RAW_PATH}")



Output directory: /content/drive/MyDrive/Dubai_Real_Estate_Data/final_pipeline_outputs/dld_social_correlation


In [ ]:
reddit_posts = pd.read_csv(f"{BASE}/Prajwal final reddit/combined_reddit_posts.csv", low_memory=False)
reddit_comments = pd.read_csv(f"{BASE}/Prajwal final reddit/combined_reddit_comments.csv", low_memory=False)

social = pd.read_csv(SOCIAL_CSV, low_memory=False)
print("Loaded social CSV:", SOCIAL_CSV)

Loaded social CSV: /content/drive/MyDrive/Dubai_Real_Estate_Data/final_pipeline_outputs/master_processed.csv


In [ ]:
len(social)

89755

Topic discovery

In [ ]:
!pip install bertopic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 4.5 MB/s eta 0:00:00


In [ ]:
social = social.dropna(subset=["clean_text"]).copy()

# Remove empty strings as well
social = social[
    social["clean_text"].str.strip() != ""
].copy()

print(len(social))

87574


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(
    social["clean_text"].tolist(),
    show_progress_bar=True,
    normalize_embeddings=True
)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/2737 [00:00<?, ?it/s]

In [ ]:
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN

umap_model = UMAP(
    n_neighbors=30,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

hdbscan_model = HDBSCAN(
    min_cluster_size=20,
    min_samples=10,
    metric="euclidean",
    prediction_data=True
)

topic_model = BERTopic(
    embedding_model=None,      # Use embeddings
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    calculate_probabilities=True,
    verbose=True
)

In [ ]:
topics, probs = topic_model.fit_transform(
    social["clean_text"].tolist(),
    embeddings
)

social["topic"] = topics

2026-07-13 11:27:37,642 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-07-13 11:44:12,242 - BERTopic - Dimensionality - Completed ✓
2026-07-13 11:44:12,248 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-07-13 12:32:21,390 - BERTopic - Cluster - Completed ✓
2026-07-13 12:32:21,478 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-07-13 12:32:28,335 - BERTopic - Representation - Completed ✓


In [ ]:
topic_info = topic_model.get_topic_info()

topic_info

,Topic,Count,Name,Representation,Representative_Docs
0,-1,41887,-1_in_to_dubai_is,"[in, to, dubai, is, for, the, and, are, of, with]",[Abu Dhabi isn’t just a “safe bet”—it’s one of...
1,0,1544,0_emaar_valley_south_heights,"[emaar, valley, south, heights, emaars, commun...",[I can offer you a ready villa in emaar south ...
2,1,841,1_distress_distressed_deals_deal,"[distress, distressed, deals, deal, below, op,...","[Is this a distress deal /s, It’s not distress..."
3,2,680,2_greens_party_labour_reform,"[greens, party, labour, reform, vote, controls...","[Its in the greens, The Greens, The Greens]"
4,3,615,3_budget_areas_rental_june,"[budget, areas, rental, june, yield, looking, ...","[Hey there, Your budget is really solid and gi..."
...,...,...,...,...,...
584,583,20,583_demand_supply_hahhahahahha_heh,"[demand, supply, hahhahahahha, heh, outstrips,...",[No. Demand and supply match up. There will al...
585,584,20,584_pharmacy_amazon_chemist_cheaper,"[pharmacy, amazon, chemist, cheaper, vaseline,...","[Cheaper than life pharmacy?, I’ve visited yes..."
586,585,20,585_evacuating_brits_brit_evacuations,"[evacuating, brits, brit, evacuations, uk, eva...","[Sell buddy sell , everyone sell , I just read..."
587,586,20,586_trustee_office_offices_trustees,"[trustee, office, offices, trustees, expire, g...",[Are the trustee offices and DLD closed? As mo...


In [ ]:
topic_model.visualize_topics()

In [ ]:
len(social)

87574

now onto topic merging and getting fewer topics and creating the SMDIs

In [ ]:
hierarchical_topics = topic_model.hierarchical_topics(
    social["clean_text"].tolist()
)

100%|██████████| 587/587 [00:04<00:00, 132.54it/s]


In [ ]:
topic_model.visualize_hierarchy(
    hierarchical_topics=hierarchical_topics
)

In [ ]:
topic_info = topic_model.get_topic_info()

topic_info.to_csv(
    "bertopic_topics.csv",
    index=False
)

In [ ]:
SAVE_DIR = "/content/drive/MyDrive/Dubai_Real_Estate_Data/SMDI creation"
import numpy as np
import pickle

# Embeddings
np.save(f"{SAVE_DIR}/embeddings.npy", embeddings)

# BERTopic model
topic_model.save(f"{SAVE_DIR}/bertopic_model")

# Data
social.to_csv(f"{SAVE_DIR}/social_with_topics.csv", index=False)

social.to_pickle(f"{SAVE_DIR}/social_with_topics.pkl")

# Topic info
topic_info.to_csv(f"{SAVE_DIR}/topic_info.csv", index=False)

# Hierarchy
with open(f"{SAVE_DIR}/hierarchical_topics.pkl", "wb") as f:
    pickle.dump(hierarchical_topics, f)

print("Everything saved successfully!")

2026-07-13 13:42:08,890 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


Everything saved successfully!


#START FROM HERE
the above code was the models which have beem exported. importing done below.

In [ ]:
!pip install bertopic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 7.8 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = "/content/drive/MyDrive/Dubai_Real_Estate_Data/SMDI creation"

Mounted at /content/drive


In [ ]:
import os

print(os.listdir(SAVE_DIR))

['embeddings.npy', 'social_with_topics.csv', 'social_with_topics.pkl', 'topic_info.csv', 'hierarchical_topics.pkl', 'bertopic_model']


In [ ]:
import numpy as np
import pandas as pd
from bertopic import BERTopic

# Load embeddings
embeddings = np.load(f"{SAVE_DIR}/embeddings.npy")

# Load BERTopic
topic_model = BERTopic.load(f"{SAVE_DIR}/bertopic_model")

# Load dataframe
social = pd.read_pickle(f"{SAVE_DIR}/social_with_topics.pkl")

# Load topic info
topic_info = pd.read_csv(f"{SAVE_DIR}/topic_info.csv")

Topic grouping

In [ ]:
topic_embeddings = topic_model.topic_embeddings_

print(topic_embeddings.shape)

(589, 384)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

topic_similarity = cosine_similarity(topic_embeddings)

topic_similarity.shape

(589, 589)

In [ ]:
topic_info[["Topic","Count","Name"]].head()

,Topic,Count,Name
0,-1,41887,-1_in_to_dubai_is
1,0,1544,0_emaar_valley_south_heights
2,1,841,1_distress_distressed_deals_deal
3,2,680,2_greens_party_labour_reform
4,3,615,3_budget_areas_rental_june


In [ ]:
from sklearn.cluster import AgglomerativeClustering

cluster_model = AgglomerativeClustering(
    n_clusters=25,
    metric="cosine",
    linkage="average"
)

groups = cluster_model.fit_predict(topic_embeddings)

topic_info["semantic_group"] = groups

In [ ]:
topic_info["semantic_group"].value_counts().sort_index()

,count
semantic_group,
0,16
1,2
2,8
3,42
4,3
5,133
6,6
7,2
8,6


In [ ]:
group = 24 #check each group and based on that create and SMDI

group_topics = topic_info[topic_info["semantic_group"] == group]

display(group_topics[["Topic", "Count", "Name"]])

,Topic,Count,Name
139,138,94,138_un_se_delulu_di
206,205,69,205_karak_samana_chai_maza
270,269,53,269_al_nahda_alr_rigga
341,340,41,340_allah_mashaallah_congratulations_inshallah
463,462,28,462_ali_jebel_hamriya_nahda
563,562,21,562_wellstudied_investorfinancial_decor_partner
